In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, Play, jslink, HBox, VBox, HTML, Layout, interactive
from IPython.display import display


# ============================================================
# ELIMINATING LIMIT CYCLES:
# QUANTIZATION STRATEGY AND LYAPUNOV ENERGY
# ============================================================
#
# This notebook compares rounding and magnitude truncation in a
# specific second-order state-space realization.
#
# The system is
#
#       q[n+1] = Q(A q[n])
#
# where
#
#                   [ cos(theta)   -sin(theta) ]
#       A = r       [                          ].
#                   [ sin(theta)    cos(theta) ]
#
# Thus A performs a rotation by theta followed by a contraction
# by the factor r.
#
# For 0 < r < 1, the corresponding unquantized system is
# asymptotically stable.
#
#
# ============================================================
# LYAPUNOV FUNCTION
# ============================================================
#
# We choose
#
#       G = I
#
# and therefore
#
#       p[n] = q^T[n] G q[n]
#            = q1^2[n] + q2^2[n].
#
# For the unquantized system,
#
#       A^T A = r^2 I,
#
# so
#
#       p[n+1] = r^2 p[n].
#
# Since r < 1,
#
#       p[n+1] < p[n]
#
# for every nonzero state.
#
#
# ============================================================
# MAGNITUDE TRUNCATION
# ============================================================
#
# Magnitude truncation is performed toward zero.
#
# Componentwise,
#
#       |Q_M(v_k)| <= |v_k|.
#
# Therefore,
#
#       ||Q_M(Aq)||_2^2 <= ||Aq||_2^2
#
# and hence
#
#       p[n+1] <= r^2 p[n] < p[n]
#
# for every nonzero state.
#
#
# ============================================================
# ROUNDING
# ============================================================
#
# Rounding does not satisfy
#
#       |Q(v_k)| <= |v_k|
#
# for every component.
#
# Near the origin, rounding may move a component away from zero.
# The strict energy-decrease mechanism may therefore be lost and
# the state may become trapped in a nonzero periodic orbit.
#
#
# ============================================================
# ANIMATION
# ============================================================
#
# The experiment starts at
#
#       n = 0.
#
# At n = 0 only the initial state q[0] is visible.
#
# The first animation step shows
#
#       q[0] -> q[1].
#
# Subsequent steps progressively construct the trajectories.
#
# ============================================================


# ------------------------------------------------------------
# Rounding quantizer: half away from zero
# ------------------------------------------------------------

def round_quantizer(x, Delta):

    x_array = np.asarray(x, dtype=float)

    index = np.where(x_array >= 0.0, np.floor(x_array / Delta + 0.5), np.ceil(x_array / Delta - 0.5))

    result = Delta * index

    if np.ndim(result) == 0:
        return float(result)

    return result


# ------------------------------------------------------------
# Magnitude truncation: toward zero
# ------------------------------------------------------------

def magnitude_truncation_quantizer(x, Delta):

    x_array = np.asarray(x, dtype=float)

    result = Delta * np.trunc(x_array / Delta)

    if np.ndim(result) == 0:
        return float(result)

    return result


# ------------------------------------------------------------
# State-transition matrix
# ------------------------------------------------------------

def state_matrix(r, theta_deg):

    theta = np.deg2rad(theta_deg)

    return r * np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])


# ------------------------------------------------------------
# Quantized state simulation
# ------------------------------------------------------------

def simulate_quantized(A, q0, Delta, iterations, quantizer):

    q = np.zeros((iterations + 1, 2), dtype=float)

    q[0, :] = quantizer(q0, Delta)

    for n in range(iterations):

        v = A @ q[n, :]

        q[n + 1, :] = quantizer(v, Delta)

    return q


# ------------------------------------------------------------
# Unquantized reference
# ------------------------------------------------------------

def simulate_unquantized(A, q0, iterations):

    q = np.zeros((iterations + 1, 2), dtype=float)

    q[0, :] = q0

    for n in range(iterations):

        q[n + 1, :] = A @ q[n, :]

    return q


# ------------------------------------------------------------
# Lyapunov energy
# ------------------------------------------------------------

def lyapunov_energy(q):

    return np.sum(q**2, axis=1)


# ------------------------------------------------------------
# Energy ratios
# ------------------------------------------------------------

def energy_ratios(p):

    if len(p) < 2:
        return np.array([])

    ratios = np.full(len(p) - 1, np.nan)

    for n in range(len(p) - 1):

        if p[n] > 1e-15:

            ratios[n] = p[n + 1] / p[n]

    return ratios


# ------------------------------------------------------------
# Detect first repeated finite-grid state
# ------------------------------------------------------------

def detect_cycle(q, Delta):

    visited = {}

    for n in range(len(q)):

        key = tuple(np.round(q[n, :] / Delta).astype(int))

        if key in visited:

            first = visited[key]

            second = n

            period = second - first

            cycle = q[first:second, :]

            return first, second, period, cycle

        visited[key] = n

    return None, None, None, np.empty((0, 2))


# ------------------------------------------------------------
# Determine whether zero state has been reached
# ------------------------------------------------------------

def zero_state_index(q):

    for n in range(len(q)):

        if np.all(np.isclose(q[n, :], 0.0)):

            return n

    return None


# ------------------------------------------------------------
# Check monotonic Lyapunov-energy decrease
# ------------------------------------------------------------

def energy_is_nonincreasing(p):

    if len(p) < 2:
        return True

    return np.all(np.diff(p) <= 1e-12)


# ============================================================
# CSS
# ============================================================

style_html = HTML("""
<style>

.le-root {
    width: 960px;
    max-width: 960px;
    font-family: Arial, sans-serif;
}

.le-header {
    background: #34323d;
    color: white;
    padding: 8px 14px;
    border-radius: 7px 7px 0 0;
    font-size: 19px;
    font-weight: bold;
}

.le-intro {
    background: #f6f6f8;
    border: 1px solid #d7d6dc;
    border-top: none;
    padding: 7px 12px;
    border-radius: 0 0 7px 7px;
    font-size: 12px;
    line-height: 1.45;
    margin-bottom: 6px;
}

.le-accent {
    font-weight: bold;
    color: #514d65;
}

.le-controls-title {
    font-size: 12.5px;
    font-weight: bold;
    margin: 0 0 3px 3px;
    color: #34323d;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Compact visible documentation
# ------------------------------------------------------------

header_html = HTML("""
<div class="le-root">

    <div class="le-header">
        Eliminating Limit Cycles: Quantization Strategy and Lyapunov Energy
    </div>

    <div class="le-intro">
        <span class="le-accent">Experiment:</span>
        compare rounding with magnitude truncation for the same stable second-order realization.
        <span class="le-accent">What to observe:</span>
        magnitude truncation forces the Lyapunov energy toward zero, while rounding may become trapped in a nonzero limit cycle.
    </div>

</div>
""")


# ============================================================
# MAIN INTERACTIVE PLOTTING FUNCTION
# ============================================================

def plot_lyapunov_lab(r=0.85, theta=45.0, K=3, q10=0.50, q20=0.25, iterations=40, time_step=0):

    Delta = 2.0**(-K)

    A = state_matrix(r, theta)

    q0 = np.array([q10, q20], dtype=float)


    # --------------------------------------------------------
    # Complete trajectories
    # --------------------------------------------------------

    q_linear_full = simulate_unquantized(A, q0, iterations)

    q_round_full = simulate_quantized(A, q0, Delta, iterations, round_quantizer)

    q_trunc_full = simulate_quantized(A, q0, Delta, iterations, magnitude_truncation_quantizer)


    # --------------------------------------------------------
    # Complete energy sequences
    # --------------------------------------------------------

    p_linear_full = lyapunov_energy(q_linear_full)

    p_round_full = lyapunov_energy(q_round_full)

    p_trunc_full = lyapunov_energy(q_trunc_full)


    # --------------------------------------------------------
    # Current time step
    # --------------------------------------------------------

    current_step = int(np.clip(time_step, 0, iterations))


    # --------------------------------------------------------
    # Visible portions
    # --------------------------------------------------------

    q_linear = q_linear_full[:current_step + 1, :]

    q_round = q_round_full[:current_step + 1, :]

    q_trunc = q_trunc_full[:current_step + 1, :]

    p_linear = p_linear_full[:current_step + 1]

    p_round = p_round_full[:current_step + 1]

    p_trunc = p_trunc_full[:current_step + 1]


    # --------------------------------------------------------
    # Cycle detection
    # --------------------------------------------------------

    round_start, round_repeat, round_period, round_cycle = detect_cycle(q_round, Delta)

    trunc_start, trunc_repeat, trunc_period, trunc_cycle = detect_cycle(q_trunc, Delta)


    # --------------------------------------------------------
    # Zero-state detection
    # --------------------------------------------------------

    round_zero = zero_state_index(q_round)

    trunc_zero = zero_state_index(q_trunc)


    # --------------------------------------------------------
    # Energy ratios
    # --------------------------------------------------------

    ratio_round = energy_ratios(p_round)

    ratio_trunc = energy_ratios(p_trunc)

    theoretical_ratio = r**2


    round_monotonic = energy_is_nonincreasing(p_round)

    trunc_monotonic = energy_is_nonincreasing(p_trunc)


    # ========================================================
    # FIGURE LAYOUT
    # ========================================================

    fig = plt.figure(figsize=(13.2, 8.1))


    grid = fig.add_gridspec(
        3,
        2,
        width_ratios=[1.25, 1.0],
        height_ratios=[1.0, 0.82, 1.08],
        wspace=0.28,
        hspace=1.03
    )


    ax_state = fig.add_subplot(grid[:, 0])

    ax_energy = fig.add_subplot(grid[0, 1])

    ax_ratio = fig.add_subplot(grid[1, 1])

    ax_monitor = fig.add_subplot(grid[2, 1])


    # ========================================================
    # PANEL 1 — STATE TRAJECTORIES
    # ========================================================

    complete_values = np.concatenate((
        q_linear_full.flatten(),
        q_round_full.flatten(),
        q_trunc_full.flatten()
    ))


    maximum_amplitude = max(
        np.max(np.abs(complete_values)),
        4.0 * Delta
    )


    limit = 1.18 * maximum_amplitude


    grid_start = np.floor(-limit / Delta) * Delta

    grid_end = np.ceil(limit / Delta) * Delta

    grid_values = np.arange(grid_start, grid_end + 0.5 * Delta, Delta)


    if len(grid_values) <= 40:

        for value in grid_values:

            ax_state.axvline(value, linewidth=0.45, alpha=0.16)

            ax_state.axhline(value, linewidth=0.45, alpha=0.16)


    # --------------------------------------------------------
    # n = 0
    # --------------------------------------------------------

    if current_step == 0:

        ax_state.plot(
            q_linear[0, 0],
            q_linear[0, 1],
            'o',
            color='tab:blue',
            markersize=5,
            label='Unquantized initial state'
        )


        ax_state.plot(
            q_round[0, 0],
            q_round[0, 1],
            'o',
            color='tab:orange',
            markersize=6,
            label='Rounding initial state'
        )


        ax_state.plot(
            q_trunc[0, 0],
            q_trunc[0, 1],
            's',
            color='tab:green',
            markersize=6,
            label='Magnitude-truncation initial state'
        )


    else:

        ax_state.plot(
            q_linear[:, 0],
            q_linear[:, 1],
            '--',
            color='tab:blue',
            linewidth=1.6,
            label='Unquantized trajectory'
        )


        ax_state.plot(
            q_round[:, 0],
            q_round[:, 1],
            'o-',
            color='tab:orange',
            linewidth=1.8,
            markersize=5.0,
            label='Rounding'
        )


        ax_state.plot(
            q_trunc[:, 0],
            q_trunc[:, 1],
            's-',
            color='tab:green',
            linewidth=1.8,
            markersize=5.0,
            label='Magnitude truncation'
        )


    # --------------------------------------------------------
    # Rounding limit cycle
    # --------------------------------------------------------

    if round_period is not None and round_zero is None and len(round_cycle) > 0:

        cycle_x = list(round_cycle[:, 0])

        cycle_y = list(round_cycle[:, 1])


        if len(round_cycle) > 1:

            cycle_x.append(cycle_x[0])

            cycle_y.append(cycle_y[0])


        ax_state.plot(
            cycle_x,
            cycle_y,
            'D-',
            color='tab:red',
            linewidth=2.5,
            markersize=7,
            label=f'Rounding period-{round_period} cycle'
        )


    # --------------------------------------------------------
    # Current states
    # --------------------------------------------------------

    if current_step > 0:

        ax_state.plot(
            q_round[-1, 0],
            q_round[-1, 1],
            'o',
            color='tab:orange',
            markerfacecolor='none',
            markersize=11,
            markeredgewidth=2.0
        )


        ax_state.plot(
            q_trunc[-1, 0],
            q_trunc[-1, 1],
            's',
            color='tab:green',
            markerfacecolor='none',
            markersize=10,
            markeredgewidth=2.0
        )


    # --------------------------------------------------------
    # Initial state
    # --------------------------------------------------------

    ax_state.plot(
        q0[0],
        q0[1],
        'D',
        color='tab:purple',
        markersize=7,
        label='Specified initial state'
    )


    # --------------------------------------------------------
    # Origin
    # --------------------------------------------------------

    ax_state.plot(
        0.0,
        0.0,
        '+',
        color='0.30',
        markersize=12,
        markeredgewidth=2.0,
        label='Origin'
    )


    ax_state.set_xlim(-limit, limit)

    ax_state.set_ylim(-limit, limit)

    ax_state.set_aspect('equal', adjustable='box')


    ax_state.set_xlabel(
        r'$q_1[n]$',
        fontsize=12,
        labelpad=9
    )


    ax_state.set_ylabel(
        r'$q_2[n]$',
        fontsize=12,
        labelpad=9
    )


    ax_state.set_title(
        f'State Evolution — Time Step n = {current_step}',
        fontsize=13
    )


    ax_state.grid(
        True,
        linestyle=':',
        alpha=0.27
    )


    ax_state.tick_params(
        labelsize=10
    )


    ax_state.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.10),
        ncol=2,
        frameon=False,
        fontsize=9.4,
        columnspacing=1.5,
        handlelength=2.0
    )


    # ========================================================
    # PANEL 2 — LYAPUNOV ENERGY
    # ========================================================

    n_energy = np.arange(current_step + 1)


    ax_energy.plot(
        n_energy,
        p_linear,
        '--o',
        color='tab:blue',
        linewidth=1.5,
        markersize=3.8,
        label='Unquantized'
    )


    ax_energy.plot(
        n_energy,
        p_round,
        'o-',
        color='tab:orange',
        linewidth=1.5,
        markersize=4.0,
        label='Rounding'
    )


    ax_energy.plot(
        n_energy,
        p_trunc,
        's-',
        color='tab:green',
        linewidth=1.5,
        markersize=4.0,
        label='Magnitude truncation'
    )


    ax_energy.set_xlim(
        0,
        iterations
    )


    energy_max = max(
        p_linear_full[0],
        p_round_full[0],
        p_trunc_full[0]
    )


    ax_energy.set_ylim(
        0.0,
        1.08 * energy_max
    )


    ax_energy.set_xlabel(
        'Iteration n',
        fontsize=10.5,
        labelpad=10
    )


    ax_energy.set_ylabel(
        r'$p[n]=\mathbf{q}^T[n]\mathbf{q}[n]$',
        fontsize=10,
        labelpad=7
    )


    ax_energy.set_title(
        'Lyapunov Energy',
        fontsize=11.5
    )


    ax_energy.tick_params(
        labelsize=9.5
    )


    ax_energy.grid(
        True,
        linestyle=':',
        alpha=0.28
    )


    ax_energy.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.37),
        ncol=3,
        frameon=False,
        fontsize=9.6,
        columnspacing=1.5,
        handlelength=1.9
    )


    # ========================================================
    # PANEL 3 — ENERGY CONTRACTION RATIO
    # ========================================================

    if len(ratio_round) > 0:

        n_ratio = np.arange(
            len(ratio_round)
        )


        ax_ratio.plot(
            n_ratio,
            ratio_round,
            'o-',
            color='tab:orange',
            linewidth=1.3,
            markersize=3.8,
            label='Rounding'
        )


        ax_ratio.plot(
            n_ratio,
            ratio_trunc,
            's-',
            color='tab:green',
            linewidth=1.3,
            markersize=3.8,
            label='Magnitude truncation'
        )


    ax_ratio.axhline(
        theoretical_ratio,
        color='tab:blue',
        linestyle='--',
        linewidth=1.4,
        label=r'Unquantized $r^2$'
    )


    ax_ratio.axhline(
        1.0,
        color='0.35',
        linestyle=':',
        linewidth=1.2,
        label='No decrease'
    )


    ax_ratio.set_xlim(
        0,
        iterations
    )


    ratio_candidates = [
        1.0,
        theoretical_ratio
    ]


    if len(ratio_round) > 0:

        ratio_candidates.extend(
            ratio_round[np.isfinite(ratio_round)]
        )


    if len(ratio_trunc) > 0:

        ratio_candidates.extend(
            ratio_trunc[np.isfinite(ratio_trunc)]
        )


    ratio_max = max(
        ratio_candidates
    )


    ax_ratio.set_ylim(
        0.0,
        max(1.10, 1.08 * ratio_max)
    )


    ax_ratio.set_xlabel(
        'Iteration n',
        fontsize=10.5,
        labelpad=10
    )


    ax_ratio.set_ylabel(
        r'$p[n+1]/p[n]$',
        fontsize=10,
        labelpad=7
    )


    ax_ratio.set_title(
        'Energy Contraction Ratio',
        fontsize=11.5
    )


    ax_ratio.tick_params(
        labelsize=9.5
    )


    ax_ratio.grid(
        True,
        linestyle=':',
        alpha=0.28
    )


    ax_ratio.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.42),
        ncol=2,
        frameon=False,
        fontsize=9.6,
        columnspacing=1.6,
        handlelength=1.9,
        labelspacing=0.8
    )


    # ========================================================
    # PANEL 4 — STATE / ENERGY MONITOR
    # ========================================================

    ax_monitor.axis(
        'off'
    )


    # --------------------------------------------------------
    # Rounding status
    # --------------------------------------------------------

    if round_zero is not None:

        rounding_status = (
            f'ZERO STATE\n'
            f'Reached at n = {round_zero}'
        )


    elif round_period is not None:

        rounding_status = (
            f'NONZERO LIMIT CYCLE\n'
            f'Start  = {round_start}\n'
            f'Period = {round_period}'
        )


    else:

        rounding_status = (
            'TRANSIENT\n'
            'No repeated state yet'
        )


    # --------------------------------------------------------
    # Magnitude-truncation status
    # --------------------------------------------------------

    if trunc_zero is not None:

        truncation_status = (
            f'ZERO STATE\n'
            f'Reached at n = {trunc_zero}'
        )


    elif trunc_period is not None:

        truncation_status = (
            f'REPEATED STATE\n'
            f'Period = {trunc_period}'
        )


    else:

        truncation_status = (
            'TRANSIENT\n'
            'Energy still decreasing'
        )


    system_text = (
        f'SYSTEM / LYAPUNOV MODEL\n'
        f'────────────────────────\n'
        f'r           : {r:.3f}\n'
        f'theta       : {theta:.1f} deg\n'
        f'K           : {K}\n'
        f'Delta       : {Delta:.6f}\n'
        f'r^2         : {theoretical_ratio:.4f}\n'
        f'G           : I\n'
        f'p[n]        : q1^2 + q2^2\n'
        f'Time n      : {current_step}'
    )


    result_text = (
        f'ROUNDING\n'
        f'──────────────────\n'
        f'{rounding_status}\n'
        f'Energy monotonic : {"YES" if round_monotonic else "NO"}\n\n'
        f'MAGNITUDE TRUNCATION\n'
        f'──────────────────\n'
        f'{truncation_status}\n'
        f'Energy monotonic : {"YES" if trunc_monotonic else "NO"}'
    )


    ax_monitor.text(
        0.01,
        0.94,
        system_text,
        transform=ax_monitor.transAxes,
        ha='left',
        va='top',
        fontsize=9.2,
        family='monospace',
        linespacing=1.40
    )


    ax_monitor.text(
        0.56,
        0.94,
        result_text,
        transform=ax_monitor.transAxes,
        ha='left',
        va='top',
        fontsize=9.1,
        family='monospace',
        linespacing=1.38
    )


    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    fig.suptitle(
        'Lyapunov-Energy View of Quantization Limit-Cycle Elimination',
        fontsize=13
    )


    plt.subplots_adjust(
        left=0.06,
        right=0.985,
        top=0.91,
        bottom=0.13
    )


    plt.show()

    plt.close(fig)


# ============================================================
# CONTROLS
# ============================================================

r_slider = FloatSlider(
    value=0.85,
    min=0.50,
    max=0.99,
    step=0.01,
    description='r:',
    continuous_update=True,
    readout_format='.2f',
    style={
        'description_width': '35px'
    },
    layout=Layout(width='330px')
)


theta_slider = FloatSlider(
    value=45.0,
    min=0.0,
    max=90.0,
    step=5.0,
    description='theta:',
    continuous_update=True,
    readout_format='.0f',
    style={
        'description_width': '50px'
    },
    layout=Layout(width='340px')
)


K_slider = IntSlider(
    value=3,
    min=2,
    max=8,
    step=1,
    description='Bits K:',
    continuous_update=True,
    style={
        'description_width': '60px'
    },
    layout=Layout(width='270px')
)


q10_slider = FloatSlider(
    value=0.50,
    min=-0.90,
    max=0.90,
    step=0.05,
    description='q1[0]:',
    continuous_update=True,
    readout_format='.2f',
    style={
        'description_width': '55px'
    },
    layout=Layout(width='320px')
)


q20_slider = FloatSlider(
    value=0.25,
    min=-0.90,
    max=0.90,
    step=0.05,
    description='q2[0]:',
    continuous_update=True,
    readout_format='.2f',
    style={
        'description_width': '55px'
    },
    layout=Layout(width='320px')
)


iterations_slider = IntSlider(
    value=40,
    min=15,
    max=80,
    step=5,
    description='Iterations:',
    continuous_update=True,
    style={
        'description_width': '75px'
    },
    layout=Layout(width='320px')
)


# ============================================================
# TIME CONTROL
# ============================================================

time_slider = IntSlider(
    value=0,
    min=0,
    max=40,
    step=1,
    description='Time step n:',
    continuous_update=True,
    style={
        'description_width': '85px'
    },
    layout=Layout(width='660px')
)


play_control = Play(
    value=0,
    min=0,
    max=40,
    step=1,
    interval=800,
    description='Play',
    disabled=False,
    layout=Layout(width='120px')
)


jslink(
    (play_control, 'value'),
    (time_slider, 'value')
)


# ============================================================
# ANIMATION LOCK
# ============================================================

animation_lock_controls = [
    r_slider,
    theta_slider,
    K_slider,
    q10_slider,
    q20_slider,
    iterations_slider,
    time_slider
]


def set_animation_lock(locked):

    for widget in animation_lock_controls:

        widget.disabled = locked


def update_animation_lock(change):

    set_animation_lock(
        bool(change['new'])
    )


play_traits = play_control.traits()


if 'playing' in play_traits:

    play_state_trait = 'playing'


elif '_playing' in play_traits:

    play_state_trait = '_playing'


else:

    play_state_trait = None


if play_state_trait is not None:

    play_control.observe(
        update_animation_lock,
        names=play_state_trait
    )


set_animation_lock(
    False
)


# ============================================================
# CONTROL SYNCHRONIZATION
# ============================================================

def update_iteration_range(change):

    new_maximum = change['new']

    time_slider.max = new_maximum

    play_control.max = new_maximum


    if time_slider.value > new_maximum:

        time_slider.value = new_maximum


    if play_control.value > new_maximum:

        play_control.value = new_maximum


iterations_slider.observe(
    update_iteration_range,
    names='value'
)


# ------------------------------------------------------------
# Parameter changes restart at n = 0
# ------------------------------------------------------------

def restart_time(change):

    time_slider.value = 0

    play_control.value = 0


r_slider.observe(
    restart_time,
    names='value'
)


theta_slider.observe(
    restart_time,
    names='value'
)


K_slider.observe(
    restart_time,
    names='value'
)


q10_slider.observe(
    restart_time,
    names='value'
)


q20_slider.observe(
    restart_time,
    names='value'
)


# ============================================================
# CONTROL LAYOUT
# ============================================================

controls_row_1 = HBox(
    [
        r_slider,
        theta_slider,
        K_slider
    ],
    layout=Layout(
        width='950px',
        justify_content='space-between',
        align_items='center'
    )
)


controls_row_2 = HBox(
    [
        q10_slider,
        q20_slider,
        iterations_slider
    ],
    layout=Layout(
        width='930px',
        justify_content='space-between',
        align_items='center'
    )
)


controls_row_3 = HBox(
    [
        play_control,
        time_slider
    ],
    layout=Layout(
        width='820px',
        justify_content='space-between',
        align_items='center'
    )
)


controls_box = VBox(
    [
        HTML("<div class='le-controls-title'>Energy experiment controls</div>"),
        controls_row_1,
        controls_row_2,
        controls_row_3
    ],
    layout=Layout(
        width='960px',
        border='1px solid #d7d6dc',
        padding='6px 8px',
        overflow='visible'
    )
)


# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_lyapunov_lab,
    r=r_slider,
    theta=theta_slider,
    K=K_slider,
    q10=q10_slider,
    q20=q20_slider,
    iterations=iterations_slider,
    time_step=time_slider
)


plot_output = widget_plot.children[-1]


plot_output.layout = Layout(
    width='auto',
    overflow='visible'
)


# ============================================================
# FINAL NOTEBOOK LAYOUT
# ============================================================

main_layout = VBox(
    [
        header_html,
        controls_box,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ============================================================
# DISPLAY
# ============================================================

display(style_html)

display(main_layout)